In [1]:
from agents import WebSearchTool,Runner,trace,function_tool,Agent
import asyncio
import os
from IPython.display import Markdown
from dotenv import load_dotenv
load_dotenv()
from agents.model_settings import ModelSettings
from pydantic import BaseModel,Field

In [2]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools = [WebSearchTool()],
    model_settings=ModelSettings(tool_choice="required")
    
)

In [89]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

Error getting response: Unknown tool type: <class 'type'>, tool. (request_id: None)


UserError: Unknown tool type: <class 'type'>, tool

Exception in thread Thread-440 (_run):
Traceback (most recent call last):
  File "/Users/sakthi/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/Users/sakthi/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/sakthi/projects_agentic/agents_raw/.venv/lib/python3.12/site-packages/agents/tracing/processors.py", line 260, in _run
    self._export_batches(force=False)
  File "/Users/sakthi/projects_agentic/agents_raw/.venv/lib/python3.12/site-packages/agents/tracing/processors.py", line 293, in _export_batches
    self._exporter.export(items_to_export)
  File "/Users/sakthi/projects_agentic/agents_raw/.venv/lib/python3.12/site-packages/agents/tracing/processors.py", line 117, in export
    response = self._client.post(url=self.endpoint, headers=headers, json=payload)
               ^^

In [3]:
how_many =3

instructions = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {how_many} terms to query for."

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=instructions,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)


message = "Latest AI Agent frameworks in 2025"

# with trace("Search"):
#     result = await Runner.run(planner_agent, message)
#     print(result.final_output)

In [4]:
@function_tool
def print_results(answer:str):
    print(answer)
    return {"status":"Done"}

In [5]:
print_results

FunctionTool(name='print_results', description='', params_json_schema={'properties': {'answer': {'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'print_results_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112642de0>, strict_json_schema=True, is_enabled=True)

In [ ]:
## So the next three functions will plan and execute the search using Planner Agent and Search Agent.
async def plan_searches(query:str):
    """Use this planner agent to plan which searches to run for the particular query."""
    print("Planning the searches")
    result = await Runner.run(planner_agent,f"Query:{query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

@function_tool
async def perform_searches(search_plan:WebSearchPlan):
    """Call search() for each item in the search plan """
    print("Something")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    result = await asyncio.gather(*tasks)
    return result
    

async def search(task:WebSearchItem):
    """I use the search_agent to run a web search for each item in the search plan"""
    input =f"search term {task.query} Reason for searching {task.reason}"
    result = await Runner.run(search_agent,input)
    return result.final_output
plan_searches_tool =planner_agent.as_tool("plan_searches",tool_description="You are an agent which is used to plan the searches to run for the particular query.So the next agent can go online or web to get the relevant data.")


In [13]:
tools = [perform_searches,plan_searches_tool]

In [17]:
INSTRUCTIONS = """You should call this print function whenever you are supplied with the answer, and it should be presented in a clean and neat email-like structure with an appropriate subject line."""

email_agent = Agent(
    name="Print_Email agent",
    instructions=INSTRUCTIONS,
    tools=[print_results],
    model="gpt-4o-mini",
)
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 100 words."
    "First, you need to call the plan_searches tool and then, after getting the searches, you need to pass the perform_searches tool and then you need to compile the report."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 1-2 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
    tools = tools
)


In [18]:
query ="Which field is more progressive in the coming years?"

with trace("Research trace with tools"):
    print("Starting research...")
    result =  await Runner.run(writer_agent,query)
    print(result)
    print("Hooray!")




Starting research...
Something
RunResult:
- Last agent: Agent(name="WriterAgent", ...)
- Final output (ReportData):
    {
      "short_summary": "The report identifies several progressive fields for the coming years, including artificial intelligence, renewable energy, healthcare technology, and cybersecurity. These sectors are driven by technological advancements, sustainability initiatives, and changing societal needs, promising significant growth and innovation opportunities.",
      "markdown_report": "# Progressive Fields for the Coming Years: A Comprehensive Analysis\n\n## 1. Introduction\nAs we advance into a new era marked by rapid technological advancements and evolving societal needs, the question of which fields will be most progressive in the upcoming years is increasingly relevant. This report explores various sectors expected to undergo significant transformations and growth due to innovations, environmental considerations, and shifting workforce demands.\n\n## 2. Industr

/var/folders/4f/04m7nfz97xn6my_gsqmc6rx00000gn/T/ipykernel_3203/2697716761.py:5: RuntimeWarning: coroutine 'Runner.run' was never awaited
  result =  await Runner.run(writer_agent,query)


In [ ]:
print(result)


In [11]:
## So the next three functions will plan and execute the search using Planner Agent and Search Agent.
async def plan_searches(query:str):
    """Use this planner agent to plan which searches to run for the particular query."""
    print("Planning the searches")
    result = await Runner.run(planner_agent,f"Query:{query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

@function_tool
async def perform_searches(search_plan:WebSearchPlan):
    """Call search() for each item in the search plan """
    print("Something")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    result = await asyncio.gather(*tasks)
    return result
    

async def search(task:WebSearchItem):
    """I use the search_agent to run a web search for each item in the search plan"""
    input =f"search term {task.query} Reason for searching {task.reason}"
    result = await Runner.run(search_agent,input)
    return result.final_output


In [8]:
print(writer_agent.tools)

[]


In [ ]:
plan_searches_tool

FunctionTool(name='plan_searches', description='You are an agent which is used to plan the searches to run for the particular query.So the next agent can go online or web to get the relevant data.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'plan_searches_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112643ba0>, strict_json_schema=True, is_enabled=True)

In [79]:
planner_agent

Agent(name='PlannerAgent', instructions='You are a helpful research assistant. Given a query, come up with a set of web searches to perform to best answer the query. Output 3 terms to query for.', handoff_description=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=<class '__main__.WebSearchPlan'>, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [9]:
# with trace("Research trace"):
#     print("Searching research..")
#     input = "Would you say which type of field will be seeing more growth in the years 2027-2030 and which will be the highest paying field?"
#     result = await Runner.run(writer_agent, input)
#     await print_results(result.final_output)  
#     print("Hooray!")

async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output
    

In [ ]:
query ="Which field is more progressive in the coming years?"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await print_results(report)  
    print("Hooray!")




Starting research...
Planning the searches
Will perform 3 searches
Something
Thinking about report...
Finished writing report


TypeError: 'FunctionTool' object is not callable

In [ ]:
import json
()

'The report analyzes the sectors expected to grow significantly in the coming years, emphasizing AI, clean energy, healthcare technology, and vast innovations influenced by emerging technologies. These trends are crucial for investment and workforce development.'

In [105]:
import textwrap

def pretty_print(text, width=80):
    wrapped = textwrap.fill(text, width=width)
    print(wrapped)
pretty_print(report.markdown_report)

# Report: Future Progressive Fields of Industry  ## Table of Contents  1.
[Introduction](#introduction) 2. [Key Sectors Poised for Growth](#key-sectors-
poised-for-growth)    - 2.1 [Artificial Intelligence (AI) and Machine
Learning](#artificial-intelligence-ai-and-machine-learning)    - 2.2 [Clean
Energy and Sustainability](#clean-energy-and-sustainability)    - 2.3
[Healthcare Technology and Biotechnology](#healthcare-technology-and-
biotechnology)    - 2.4 [Software Publishing](#software-publishing)    - 2.5
[Renewable Energy](#renewable-energy)    - 2.6 [Hybrid and Electric Vehicle
Manufacturing](#hybrid-and-electric-vehicle-manufacturing)    - 2.7 [3D Printing
and Rapid Prototyping Services](#3d-printing-and-rapid-prototyping-services)
- 2.8 [Semiconductor and Electronic Parts Manufacturing](#semiconductor-and-
electronic-parts-manufacturing)    - 2.9 [Biotechnology](#biotechnology)    -
2.10 [Accounting Services](#accounting-services) 3. [Emerging Technologies
Influencing Growth](

In [20]:
from os import name
from pydantic import BaseModel, Field
from agents import Agent
from typing import Literal

no_of_searches = 3

class WebResults(BaseModel):
    query:str = Field(description="Questions")
    reason:str= Field(description="Reason why we want to search this query")

class WebResultPlan(BaseModel):
    status:Literal["ask_user","research"]
    questions:list[str] = Field(default_factory=list)
    missing_fields:list[str] = Field(default_factory=list)
    searches:list[WebResults] = Field(description="The actual plan goes here.")



planner_agent = Agent(name="Planning Agent",instructions="After the user gives the query, you need to plan out how you will proceed." 
"Either you need to give the additional query and the reason why you chose that query for the deep research purpose"
"Or if you want any additional information from the user end, you need to ask the questions and set the status as Ask User."
 "You can ask what and all fields you need in the missing field section. If you want, you can use the missing field section.",
model="gpt-4o-mini",
output_type=WebResultPlan)




planner_agent.as_tool("dw",tool_description="Ji")

FunctionTool(name='dw', description='Ji', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'dw_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x113703d80>, strict_json_schema=True, is_enabled=True)